# Explainable Dysgraphia Index: walkthrough

A stage-by-stage tour of the pipeline in `edi/`. Every function here is imported
from the package rather than redefined, so this notebook and `run.py` share one
implementation and cannot drift apart.

To run the whole study end to end, skip to the last cell.

**Paper.** V. Sreekumar, A. P. Anbumunee and S. Senthilkumar, "Image-Based
Dysgraphia Detection with Explainability Using CAMs and EDI," *2026 International
Conference on Computing, Communication, Control and Cyber-Physical Systems
(I5CPS)*, pp. 1-6. DOI [10.1109/I5CPS67958.2026.11452617](https://doi.org/10.1109/I5CPS67958.2026.11452617).

## Setup

On Colab, set the runtime to a GPU first (Runtime > Change runtime type > T4
GPU), then clone the repository and install the one missing package. Colab
already ships CUDA builds of PyTorch and torchvision plus most of the
scientific stack, so do **not** run `pip install -r requirements.txt`: on Linux
it resolves to CPU wheels for torch and would move the entire run onto the CPU.

Locally, run this notebook from the repository root and leave `IS_COLAB` false.

In [ ]:
# Colab only. Locally, leave IS_COLAB = False.
# On Colab: Runtime > Change runtime type > T4 GPU, first.
IS_COLAB = False

if IS_COLAB:
    # Colab provides CUDA torch/torchvision and most of the scientific stack.
    # Installing requirements.txt here would replace torch with CPU wheels.
    !git clone --quiet https://github.com/Aadhavann/Dysgraphia-Detection.git
    %cd Dysgraphia-Detection
    !pip install -q seaborn

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
print("repository root:", REPO_ROOT)

## Configuration

One dataclass holds every tunable, so a config object fully identifies a run.
`stability_runs` controls how many models are trained from different seeds; two
or more enables a measured CAM stability score.

In [ ]:
from edi import PipelineConfig

config = PipelineConfig(
    data_dir=REPO_ROOT / "data",
    results_dir=REPO_ROOT / "results",
    model_type="resnet18",
    epochs=50,
    batch_size=16,
    seed=42,
    scorecam_channels=32,
    stability_runs=2,
)
config

## Stage 1. Load the data and split it

The dataset is the *Potential Dysgraphia Handwriting Dataset of School-Age
Children* (Ramlan, 2023, Mendeley Data, CC BY 4.0): 249 scanned notebook pages,
135 low-potential and 114 potential dysgraphia.

`writer_grouped_split` partitions by subject so that no writer appears in two
splits. It returns a `SplitReport` describing what it actually did.

**Read the report.** This dataset ships no writer identifiers, so every page
resolves to its own subject and the partition is equivalent to a random
per-image split. The report detects that and warns about it rather than
describing the split as writer-independent. Supplying
`load_dataset(..., writer_map=...)` with real writer codes is what would make
the guarantee hold.

In [ ]:
from edi.data import load_dataset, writer_grouped_split

image_paths, labels, subjects = load_dataset(config.data_dir)
print(f"images: {len(image_paths)}  control: {(labels == 0).sum()}  dysgraphia: {(labels == 1).sum()}")

train_split, val_split, test_split, report = writer_grouped_split(
    image_paths, labels, subjects,
    test_size=config.test_size, val_size=config.val_size, seed=config.seed,
)
print()
print(report.as_text())

## Stage 2. The classifier

A ResNet-18 with a two-class head. Forward and backward hooks on the last
convolutional block cache activations and gradients, which is what the CAM
methods read.

In [ ]:
import torch
from edi.model import DysgraphiaClassifier, describe
from edi.train import resolve_device

device = resolve_device()
model = DysgraphiaClassifier(model_type=config.model_type).to(device)
print(f"device: {device}")
for line in describe(model):
    print(" ", line)

## Stage 3. Train

Early stopping and model selection both use validation AUC rather than
accuracy, because AUC is the metric the paper reports and the two classes are
not exactly balanced. The best checkpoint is reloaded before returning.

In [ ]:
from torch.utils.data import DataLoader
from edi.data import DysgraphiaDataset
from edi.pipeline import build_transforms
from edi.train import set_seed, train_classifier

set_seed(config.seed)

train_loader = DataLoader(
    DysgraphiaDataset(*train_split, build_transforms(augment=True)),
    batch_size=config.batch_size, shuffle=True, num_workers=0,
)
val_loader = DataLoader(
    DysgraphiaDataset(*val_split, build_transforms(augment=False)),
    batch_size=config.batch_size, shuffle=False, num_workers=0,
)
test_loader = DataLoader(
    DysgraphiaDataset(*test_split, build_transforms(augment=False)),
    batch_size=1, shuffle=False, num_workers=0,   # batch size 1: CAM is per image
)

model, history = train_classifier(
    model, train_loader, val_loader, device, config,
    checkpoint_path=config.results_dir / "checkpoints" / f"model_seed{config.seed}.pth",
)
print(f"\nbest validation AUC {history.best_val_auc:.4f} at epoch {history.best_epoch}")

## Stage 4. Score-CAM

Score-CAM masks the input with each channel's upsampled activation map and
weights the channel by the resulting gain in confidence for the predicted class.
It needs no gradients, which makes its heatmaps more stable across runs than
Grad-CAM, at the cost of `scorecam_channels` extra forward passes per image.

Both methods return a heatmap normalised to the same range, so nothing
downstream needs to know which one produced it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from edi.cam import generate_gradcam, generate_scorecam
from edi.visualize import overlay_heatmap

images, _, paths = next(iter(test_loader))
tensor = images[0:1].to(device)
predicted = int(model(tensor).argmax(dim=1).item())

import cv2
page = cv2.cvtColor(cv2.imread(str(paths[0])), cv2.COLOR_BGR2RGB)

score_cam = generate_scorecam(model, tensor, predicted, device, num_channels=config.scorecam_channels)
grad_cam = generate_gradcam(model, tensor, predicted, device)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(page); axes[0].set_title("page"); axes[0].axis("off")
axes[1].imshow(overlay_heatmap(page, score_cam.cpu().numpy().squeeze())); axes[1].set_title("Score-CAM overlay"); axes[1].axis("off")
if grad_cam is not None:
    axes[2].imshow(overlay_heatmap(page, grad_cam.cpu().numpy().squeeze())); axes[2].set_title("Grad-CAM overlay"); axes[2].axis("off")
plt.tight_layout(); plt.show()

## Stage 5. Metrics and the index

Each of the four clinical metrics is computed twice, once over the whole page
and once over the page weighted by the activation map. The difference between
the two is the interpretable part: it says whether attention landed on the
regions that carry the signal.

Every row also records the classifier's softmax probability for the positive
class, not just its hard label. That is what makes a genuine ROC-AUC and a
genuine late fusion possible.

In [ ]:
from edi.features import compute_edi_features, construct_edi

features_df = compute_edi_features(
    model, test_loader, device,
    cam_method="scorecam", num_channels=config.scorecam_channels,
)
features_df, scaler = construct_edi(features_df, low_variance_eps=config.low_variance_eps)

print(f"test images with features: {len(features_df)}")
features_df[["label", "prediction", "cnn_prob", "EDI_equal", "EDI_weighted"]].head()

## Stage 6. Evaluate

No head is scored against a label it was fitted to.

`cnn_only` uses the stored softmax probabilities and fits nothing. Every other
head is fitted with `StratifiedKFold` `cross_val_predict`, so each probability
comes from a fold in which that row was held out. Confidence intervals are 95%
percentile bootstraps over 2000 resamples, because with a test partition of
about fifty images a two-decimal AUC quoted on its own overstates the precision
available.

Fitting and scoring on the same rows reaches a perfect 1.000 AUC on pure noise.
`tests/test_evaluate.py` demonstrates that failure mode and asserts this
implementation does not reproduce it.

In [ ]:
from edi.evaluate import evaluate_all, fusion_improves_on_cnn, summarise

results = evaluate_all(features_df, cv_folds=config.cv_folds, seed=config.seed)
table = summarise(results)
table

In [ ]:
delta = fusion_improves_on_cnn(results)
print(f"fusion minus CNN-only AUC: {delta:+.4f}")
print()
print("A negative delta is a legitimate outcome and is reported as-is.")
print("It means the attention-weighted metrics added nothing the CNN had not")
print("already captured on this partition, which is worth knowing either way.")

## Stage 7. CAM stability

Stability is the mean intersection-over-union between heatmaps produced by
independently trained models on the same images. Two models are the minimum.

Earlier revisions of this pipeline returned a hard-coded `0.5` here instead of
measuring anything. This function computes it, and reports `None` when it cannot
rather than substituting a placeholder.

In [ ]:
from edi.cam import cam_stability_iou
from edi.train import set_seed, train_classifier

second = DysgraphiaClassifier(model_type=config.model_type).to(device)
set_seed(config.seed + 1)
second, _ = train_classifier(
    second, train_loader, val_loader, device, config,
    checkpoint_path=config.results_dir / "checkpoints" / f"model_seed{config.seed + 1}.pth",
    verbose=False,
)

test_tensors = [images[0:1].to(device) for images, _, _ in test_loader]
stability = cam_stability_iou(
    [model, second], test_tensors, device,
    method="scorecam", threshold=config.cam_threshold, num_channels=config.scorecam_channels,
)
stability

## Run the whole study instead

`run_pipeline` performs every stage above and writes `edi_features.csv`,
`results_table.csv`, `results_summary.txt`, `run_manifest.json`, the trained
weights and all figures to `results/`.

The manifest records the full config, the environment, the partition sizes and
every metric, so a run is identifiable from that file alone.

From a shell, the same thing is `python run.py`.

In [ ]:
from edi import run_pipeline

outcome = run_pipeline(config, cam_method="scorecam")
print(outcome["summary_table"].to_string(index=False))
print()
print("CAM stability:", outcome["stability"])